# Module 06: OpenCV for Computer Vision & Machine Learning
## Notebook 05: Feature Detection, Description, and Matching

Global image matching is brittle to occlusion, translation, rotation, and viewpoint changes. Modern computer vision relies on **Local Invariant Features**: identifying distinctive interest points (keypoints), describing their local appearance with invariant descriptor vectors, and matching them across arbitrary cluttered scenes.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Understand the mathematical criteria for corner detection via the **Harris Corner Detector** and the **Shi-Tomasi (Good Features to Track)** detector.
2. Extract fast, rotation-invariant, patent-free features using **ORB (Oriented FAST and Rotated BRIEF)** with `cv2.ORB_create()`.
3. Compute and inspect binary feature descriptors.
4. Match descriptors across images using **Brute-Force Matcher** (`cv2.BFMatcher`) and **FLANN-based Matcher** (`cv2.FlannBasedMatcher`).
5. Filter false matches using **Lowe's Ratio Test** on $k$-Nearest Neighbors ($k=2$).
6. **Advanced:** Localize an arbitrary target object inside a cluttered scene by computing a robust **Homography Matrix via RANSAC** (`cv2.findHomography` with `cv2.RANSAC`).

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_dir = "images" if os.path.exists("images") else "../images"
print(f"OpenCV Version: {cv2.__version__}")

### 1. Interest Point Detection: Harris & Shi-Tomasi Corners
Corners are ideal interest points because their gradient varies significantly in all spatial directions:
- **Structure Tensor ($M$):**
  $$M = \sum_{(u, v) \in W} w(u, v) \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$
- **Harris Corner Response ($R$):**
  $$R = \det(M) - k \cdot (\text{Tr}(M))^2 = \lambda_1 \lambda_2 - k (\lambda_1 + \lambda_2)^2$$
  - $R > 0$: Corner region (both eigenvalues $\lambda_1, \lambda_2$ large).
  - $R < 0$: Edge region (one large eigenvalue).
  - $|R|$ small: Flat region.
- **Shi-Tomasi Detector (`cv2.goodFeaturesToTrack`):** Uses the simplified score $R = \min(\lambda_1, \lambda_2) > \lambda_{\min}$, providing superior tracking stability.

In [ ]:
# Load query object
query_bgr = cv2.imread(os.path.join(img_dir, "query_object.png"))
query_gray = cv2.cvtColor(query_bgr, cv2.COLOR_BGR2GRAY)

# 1. Harris Corner Detection
harris_resp = cv2.cornerHarris(query_gray, blockSize=2, ksize=3, k=0.04)
# Dilate response for visualization
harris_dilated = cv2.dilate(harris_resp, None)
harris_vis = query_bgr.copy()
harris_vis[harris_dilated > 0.01 * harris_dilated.max()] = [0, 0, 255] # Red marks

# 2. Shi-Tomasi (Good Features to Track)
corners = cv2.goodFeaturesToTrack(query_gray, maxCorners=40, qualityLevel=0.05, minDistance=10)
shi_vis = query_bgr.copy()
if corners is not None:
    for pt in corners:
        x, y = pt.ravel()
        cv2.circle(shi_vis, (int(x), int(y)), 4, (0, 255, 0), -1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].imshow(cv2.cvtColor(harris_vis, cv2.COLOR_BGR2RGB))
axes[0].set_title("Harris Corner Detector (Red Points)")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(shi_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title("Shi-Tomasi Good Features to Track (Green Points)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### 2. Modern Invariant Feature Extraction: ORB
ORB (Oriented FAST and Rotated BRIEF) is an open-source, patent-free alternative to SIFT and SURF:
1. **Keypoint Detection:** Uses **FAST** (Features from Accelerated Segment Test) across multi-scale pyramids to achieve scale invariance.
2. **Keypoint Orientation:** Computes intensity centroid of the corner patch; orientation is vector from corner center to centroid:
   $$\theta = \arctan2(m_{01}, m_{10})$$
3. **Descriptor Generation:** Uses **Rotated BRIEF** to produce a 256-bit (32-byte) binary descriptor vector steered according to keypoint orientation.
- Matching binary descriptors uses the extremely fast **Hamming Distance** (hardware bitwise XOR and POPCOUNT).

In [ ]:
# Load Query Object and Cluttered Scene
scene_bgr = cv2.imread(os.path.join(img_dir, "scene_search.jpg"))
scene_gray = cv2.cvtColor(scene_bgr, cv2.COLOR_BGR2GRAY)

# Initialize ORB detector
orb = cv2.ORB_create(nfeatures=1500, scaleFactor=1.2, nlevels=8)

# Detect keypoints and compute descriptors
kp_query, des_query = orb.detectAndCompute(query_gray, None)
kp_scene, des_scene = orb.detectAndCompute(scene_gray, None)

print(f"Query Image: {len(kp_query)} keypoints, Descriptors shape: {des_query.shape}")
print(f"Scene Image: {len(kp_scene)} keypoints, Descriptors shape: {des_scene.shape}")

# Visualize keypoints with scale and orientation vectors
vis_kp_query = cv2.drawKeypoints(query_bgr, kp_query, None, color=(0, 255, 0),
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis_kp_scene = cv2.drawKeypoints(scene_bgr, kp_scene, None, color=(0, 255, 0),
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(vis_kp_query, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Query Object ORB Keypoints (N={len(kp_query)})")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(vis_kp_scene, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Scene Search Image ORB Keypoints (N={len(kp_scene)})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### 3. Descriptor Matching & Lowe's Ratio Test
Matching finds the closest descriptor in the scene for each query descriptor:
- For binary descriptors (ORB), use **`cv2.NORM_HAMMING`**.
- **Brute-Force Matcher (`cv2.BFMatcher`):** Exhaustively compares all descriptor pairs.
- **FLANN-based Matcher (`cv2.FlannBasedMatcher`):** Approximate Nearest Neighbors; uses Multi-Probe LSH (Locality Sensitive Hashing) for fast matching on massive image collections.
- **Lowe's Ratio Test:** Ambiguous or repetitive textures match multiple scene locations with almost equal distances. By finding the 2 nearest neighbors ($k=2$, returning matches $m$ and $n$ with distances $d_1, d_2$), David Lowe's criterion keeps only matches where:
  $$\frac{d_1}{d_2} < \tau \quad (\text{typically } \tau = 0.75)$$

In [ ]:
# 1. Create BFMatcher with Hamming distance
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

# 2. Find 2 nearest neighbors for each query descriptor
matches_knn = bf.knnMatch(des_query, des_scene, k=2)

# 3. Apply Lowe's Ratio Test
ratio_thresh = 0.75
good_matches = []
for m, n in matches_knn:
    if m.distance < ratio_thresh * n.distance:
        good_matches.append(m)

print(f"Total raw matches: {len(matches_knn)}")
print(f"High-confidence matches passing Lowe's ratio test: {len(good_matches)}")

# Visualize the filtered good matches
img_matches = cv2.drawMatches(
    query_bgr, kp_query,
    scene_bgr, kp_scene,
    good_matches[:35], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(16, 7))
plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
plt.title(f"ORB Keypoint Matches Filtered with Lowe's Ratio Test (< 0.75)")
plt.axis("off")
plt.show()

### 4. Complex Application: Robust Object Localization with RANSAC Homography
Even after Lowe's ratio test, background clutter can produce spurious outlier matches.
To localize the exact 2D orientation and position of the object:
1. Extract $(x, y)$ coordinates of matched keypoints in both images: `src_pts` and `dst_pts`.
2. Compute the Homography matrix $H$ using **RANSAC (Random Sample Consensus)** via `cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)`.
   - RANSAC randomly samples 4 point pairs, computes candidate $H$, tests all remaining points, and identifies true inliers while discarding outliers.
3. Transform the 4 outer corners of the query object into the scene space using `cv2.perspectiveTransform(corners, H)`.
4. Draw the detected bounding quadrilateral directly onto the cluttered scene.

In [ ]:
# Ensure we have sufficient inlier candidates (minimum 4 required for homography)
MIN_MATCH_COUNT = 10

if len(good_matches) >= MIN_MATCH_COUNT:
    src_pts = np.float32([kp_query[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Compute Homography with RANSAC outlier rejection
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    matches_mask = mask.ravel().tolist()
    num_inliers = np.sum(matches_mask)
    print(f"RANSAC Inliers: {num_inliers} / {len(good_matches)} ({100 * num_inliers / len(good_matches):.1f}%)")

    # Get query object dimensions and 4 boundary corners
    h_q, w_q = query_gray.shape
    query_corners = np.float32([
        [0, 0],
        [w_q - 1, 0],
        [w_q - 1, h_q - 1],
        [0, h_q - 1]
    ]).reshape(-1, 1, 2)

    # Project query corners into scene coordinate space
    scene_corners = cv2.perspectiveTransform(query_corners, H)

    # Draw detected polygon outline on the scene
    detected_scene = scene_bgr.copy()
    cv2.polylines(detected_scene, [np.int32(scene_corners)], isClosed=True, color=(0, 255, 0), thickness=4, lineType=cv2.LINE_AA)
    
    # Draw inlier matches visualization
    draw_params = dict(
        matchColor=(0, 255, 0),       # Draw inliers in green
        singlePointColor=None,
        matchesMask=matches_mask,      # Only draw inlier matches
        flags=2
    )
    img_homography = cv2.drawMatches(query_bgr, kp_query, detected_scene, kp_scene, good_matches, None, **draw_params)

    plt.figure(figsize=(16, 7))
    plt.imshow(cv2.cvtColor(img_homography, cv2.COLOR_BGR2RGB))
    plt.title(f"Target Object Successfully Localized in Cluttered Scene via RANSAC Homography ({num_inliers} Inliers)")
    plt.axis("off")
    plt.show()
else:
    print(f"Not enough good matches found: {len(good_matches)}/{MIN_MATCH_COUNT}")